In [36]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

In [37]:
train=pd.read_csv("train.csv")
test=pd.read_csv("test.csv")
test_labels=pd.read_csv("test_labels.csv")

In [38]:
train.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0000997932d777bf,Explanation\nWhy the edits made under my usern...,0,0,0,0,0,0
1,000103f0d9cfb60f,D'aww! He matches this background colour I'm s...,0,0,0,0,0,0
2,000113f07ec002fd,"Hey man, I'm really not trying to edit war. It...",0,0,0,0,0,0
3,0001b41b1c6bb37e,"""\nMore\nI can't make any real suggestions on ...",0,0,0,0,0,0
4,0001d958c54c6e35,"You, sir, are my hero. Any chance you remember...",0,0,0,0,0,0


In [39]:
test.head()

,id,comment_text
0,00001cee341fdb12,Yo bitch Ja Rule is more succesful then you'll...
1,0000247867823ef7,== From RfC == \n\n The title is fine as it is...
2,00013b17ad220c46,""" \n\n == Sources == \n\n * Zawe Ashton on Lap..."
3,00017563c3f7919a,":If you have a look back at the source, the in..."
4,00017695ad8997eb,I don't anonymously edit articles at all.


In [40]:
test_labels.head()

,id,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,00001cee341fdb12,-1,-1,-1,-1,-1,-1
1,0000247867823ef7,-1,-1,-1,-1,-1,-1
2,00013b17ad220c46,-1,-1,-1,-1,-1,-1
3,00017563c3f7919a,-1,-1,-1,-1,-1,-1
4,00017695ad8997eb,-1,-1,-1,-1,-1,-1


In [41]:
test_df=test.merge(test_labels,on="id",how="inner")
test_df = test_df[test_df["toxic"]!=-1].reset_index(drop=True)
test_df.head()

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,0001ea8717f6de06,Thank you for understanding. I think very high...,0,0,0,0,0,0
1,000247e83dcc1211,:Dear god this site is horrible.,0,0,0,0,0,0
2,0002f87b16116a7f,"""::: Somebody will invariably try to add Relig...",0,0,0,0,0,0
3,0003e1cccfd5a40a,""" \n\n It says it right there that it IS a typ...",0,0,0,0,0,0
4,00059ace3e3e9a53,""" \n\n == Before adding a new product to the l...",0,0,0,0,0,0


In [42]:
label_cols = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

train["is_toxic"]=(train[label_cols].sum(axis=1)>0).astype(int)
test_df["is_toxic"]=(test_df[label_cols].sum(axis=1)>0).astype(int)

print("Train label distribution:")
print(train["is_toxic"].value_counts())
print('\n')
print("Test label distribution:")
print(test_df["is_toxic"].value_counts())

Train label distribution:
is_toxic
0    143346
1     16225
Name: count, dtype: int64


Test label distribution:
is_toxic
0    57735
1     6243
Name: count, dtype: int64


In [43]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'[^a-zA-Z ]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train["clean_text"]   = train["comment_text"].apply(clean_text)
test_df["clean_text"] = test_df["comment_text"].apply(clean_text)
print("Sample cleaned text:", train["clean_text"].iloc[0][:120])

Sample cleaned text: explanation why the edits made under my username hardcore metallica fan were reverted they weren t vandalisms just closu


In [25]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

scale_pos = int((df["is_toxic"]==0).sum()/(df["is_toxic"]==1).sum())
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    scale_pos_weight=scale_pos,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42
)

X_train_xgb, X_val_xgb, y_train_xgb, y_val_xgb=train_test_split(train_embeddings, df["is_toxic"], test_size=0.2,
                                                                random_state=42, stratify=df["is_toxic"])
xgb_model.fit(X_train_xgb, y_train_xgb)
xgb_preds=xgb_model.predict(test_embeddings)
xgb_preds_prob=xgb_model.predict_proba(test_embeddings)[:, 1]

print("XGBoost(test set)")
print(classification_report(test_df["is_toxic"], xgb_preds, target_names=["clean", "toxic"]))
print(f"ROC-AUC: {roc_auc_score(test_df['is_toxic'], xgb_preds_prob):.4f}")

XGBoost(test set)
              precision    recall  f1-score   support

       clean       0.96      0.95      0.96     57735
       toxic       0.58      0.61      0.60      6243

    accuracy                           0.92     63978
   macro avg       0.77      0.78      0.78     63978
weighted avg       0.92      0.92      0.92     63978

ROC-AUC: 0.9288


In [44]:
from sentence_transformers import SentenceTransformer

embedder=SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
sample_size=20000
df=train.sample(sample_size, random_state=42).reset_index(drop=True)
test_df=test_df.reset_index(drop=True)

train_embeddings=embedder.encode(df["clean_text"].tolist(),batch_size=64, show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

In [45]:
test_embeddings=embedder.encode(test_df["clean_text"].tolist(),batch_size=64, show_progress_bar=True)

print("Train embeddings shape:", train_embeddings.shape)
print("Test  embeddings shape:", test_embeddings.shape)

Batches:   0%|          | 0/1000 [00:00<?, ?it/s]

Train embeddings shape: (20000, 384)
Test  embeddings shape: (63978, 384)


In [46]:
from sklearn.neighbors import NearestNeighbors

embeddings=train_embeddings
knn=NearestNeighbors(n_neighbors=10, metric="cosine")
knn.fit(embeddings)
distances, neighbors=knn.kneighbors(embeddings)

edge_list = []
for i, (nbrs, dists) in enumerate(zip(neighbors, distances)):
    for nbr, dist in zip(nbrs, dists):
        if nbr!=i:
            edge_list.append([i, nbr])

print(f"Nodes: {len(embeddings)}|Edges: {len(edge_list)}")

Nodes: 20000|Edges: 180000


In [47]:
import torch
from torch_geometric.data import Data
x=torch.tensor(embeddings, dtype=torch.float)
edge_index=torch.tensor(np.array(edge_list).T, dtype=torch.long)
y=torch.tensor(df["is_toxic"].values, dtype=torch.long)

graph=Data(x=x, edge_index=edge_index, y=y)
print(graph)

Data(x=[20000, 384], edge_index=[2, 180000], y=[20000])


In [48]:
num_nodes=graph.num_nodes
perm=torch.randperm(num_nodes)
train_size=int(0.8 * num_nodes)

train_mask=torch.zeros(num_nodes, dtype=torch.bool)
val_mask=torch.zeros(num_nodes, dtype=torch.bool)
train_mask[perm[:train_size]]=True
val_mask[perm[train_size:]]=True

graph.train_mask=train_mask
graph.val_mask=val_mask
print(f"Train nodes: {train_mask.sum().item()}|Val nodes: {val_mask.sum().item()}")

Train nodes: 16000|Val nodes: 4000


In [49]:
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class GraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, dropout=0.3):
        super().__init__()
        self.conv1=SAGEConv(in_channels, hidden_channels)
        self.conv2=SAGEConv(hidden_channels, out_channels)
        self.dropout=dropout

    def forward(self,x,edge_index):
        x=self.conv1(x,edge_index)
        x=F.relu(x)
        x=F.dropout(x,p=self.dropout,training=self.training)
        x=self.conv2(x,edge_index)
        return x

device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
model=GraphSAGE(in_channels=384, hidden_channels=128, out_channels=2).to(device)
graph=graph.to(device)
print(model)
print(f"Device: {device}")


GraphSAGE(
  (conv1): SAGEConv(384, 128, aggr=mean)
  (conv2): SAGEConv(128, 2, aggr=mean)
)
Device: cpu


In [50]:
#Training GraphSAGE for 100 epochs with weighted cross-entropy for class imbalance
from torch.optim import Adam
from sklearn.metrics import f1_score

toxic_count=df["is_toxic"].sum()
clean_count=len(df)-toxic_count
class_weights=torch.tensor([1.0, clean_count/toxic_count], dtype=torch.float).to(device)

optimizer=Adam(model.parameters(), lr=0.005, weight_decay=5e-4)
criterion=nn.CrossEntropyLoss(weight=class_weights)

def train_step():
    model.train()
    optimizer.zero_grad()
    out=model(graph.x, graph.edge_index)
    loss=criterion(out[graph.train_mask], graph.y[graph.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

def val_step():
    model.eval()
    with torch.no_grad():
        out=model(graph.x, graph.edge_index)
        preds=out.argmax(dim=1)
        y_val=graph.y[graph.val_mask].cpu().numpy()
        p_val=preds[graph.val_mask].cpu().numpy()
        f1=f1_score(y_val, p_val, average="macro")
    return f1

best_f1=0
for epoch in range(1, 101):
    loss=train_step()
    f1=val_step()
    if f1>best_f1:
        best_f1=f1
        torch.save(model.state_dict(), "best_sage.pt")
    if epoch % 10==0:
        print(f"Epoch {epoch:3d}|Loss: {loss:.4f}|Val macro-F1: {f1:.4f}")

print(f"\nBest val macro F1: {best_f1:.4f}")

Epoch  10|Loss: 0.3683|Val macro-F1: 0.7099
Epoch  20|Loss: 0.2845|Val macro-F1: 0.7377
Epoch  30|Loss: 0.2398|Val macro-F1: 0.7663
Epoch  40|Loss: 0.2149|Val macro-F1: 0.7834
Epoch  50|Loss: 0.2004|Val macro-F1: 0.7915
Epoch  60|Loss: 0.1889|Val macro-F1: 0.7911
Epoch  70|Loss: 0.1836|Val macro-F1: 0.7760
Epoch  80|Loss: 0.1745|Val macro-F1: 0.7909
Epoch  90|Loss: 0.1679|Val macro-F1: 0.7907
Epoch 100|Loss: 0.1656|Val macro-F1: 0.8014

Best val macro F1: 0.8014


In [32]:
#Best GraphSAGE checkpoint on validation split
from sklearn.metrics import classification_report as cr

model.load_state_dict(torch.load("best_sage.pt", map_location=device))
model.eval()
with torch.no_grad():
    out=model(graph.x, graph.edge_index)
    probs=torch.softmax(out, dim=1)[:, 1].cpu().numpy()
    preds=out.argmax(dim=1).cpu().numpy()

y_val=graph.y[graph.val_mask.cpu()].cpu().numpy()
p_val=preds[graph.val_mask.cpu().numpy()]
pr_val=probs[graph.val_mask.cpu().numpy()]

print("GraphSAGE (val split)")
print(cr(y_val, p_val, target_names=["clean","toxic"]))
print(f"ROC-AUC: {roc_auc_score(y_val, pr_val):.4f}")

GraphSAGE (val split)
              precision    recall  f1-score   support

       clean       0.98      0.92      0.95      3631
       toxic       0.53      0.85      0.65       369

    accuracy                           0.92      4000
   macro avg       0.76      0.88      0.80      4000
weighted avg       0.94      0.92      0.92      4000

ROC-AUC: 0.9530


In [51]:
# Comparing both XGBoost and GraphSAGE on the same validation nodes
from sklearn.metrics import accuracy_score, precision_score, recall_score

# GraphSAGE val nodes
val_idx=graph.val_mask.cpu().numpy()
gs_labels=df["is_toxic"].values[val_idx]
gs_preds=preds[val_idx]

# XGBoost — use its own held-out split (never seen during training)
xgb_val_preds=xgb_model.predict(X_val_xgb)
xgb_labels=y_val_xgb.values  # ground truth for XGBoost's val set

results=pd.DataFrame({
    "Metric": ["Accuracy", "Precision (toxic)", "Recall (toxic)", "Macro F1"],
    "XGBoost": [
        accuracy_score(xgb_labels,  xgb_val_preds),
        precision_score(xgb_labels, xgb_val_preds, pos_label=1, zero_division=0),
        recall_score(xgb_labels,    xgb_val_preds, pos_label=1),
        f1_score(xgb_labels,        xgb_val_preds, average="macro")
    ],
    "GraphSAGE": [
        accuracy_score(gs_labels,  gs_preds),
        precision_score(gs_labels, gs_preds, pos_label=1, zero_division=0),
        recall_score(gs_labels,    gs_preds, pos_label=1),
        f1_score(gs_labels,        gs_preds, average="macro")
    ]
}).set_index("Metric")
results=results.round(4)
print(results.to_string())

                   XGBoost  GraphSAGE
Metric                               
Accuracy            0.9488     0.9282
Precision (toxic)   0.8255     0.5832
Recall (toxic)      0.6165     0.8717
Macro F1            0.8389     0.8291


In [52]:
#building a NetworkX graph and Louvain community detection
import networkx as nx
import community as community_louvain

G=nx.Graph()
G.add_nodes_from(range(len(df)))
G.add_edges_from(edge_list)

partition=community_louvain.best_partition(G, random_state=42)
df["community"]=df.index.map(partition)

num_communities=df["community"].nunique()
print(f"Number of communities detected={num_communities}")
print(df["community"].value_counts().head(10))

Number of communities detected=23
community
7     2697
1     2056
5     1773
14    1754
17    1568
13    1314
4     1269
22     976
11     887
9      834
Name: count, dtype: int64


In [53]:
community_stats = (
    df.groupby("community")
      .agg(
          total_comments=("is_toxic", "count"),toxic_comments=("is_toxic","sum")
      )
      .assign(toxicity_rate = lambda x: x["toxic_comments"] / x["total_comments"])
      .sort_values("toxicity_rate", ascending=False)
      .reset_index()
)
TOXIC_THRESHOLD=0.40
suspicious=community_stats[community_stats["toxicity_rate"] >= TOXIC_THRESHOLD]

print(f"Suspicious communities (toxicity rate>={TOXIC_THRESHOLD*100:.0f}%): {len(suspicious)}")
print('\n')
print(suspicious.to_string(index=False))

Suspicious communities (toxicity rate>=40%): 1


 community  total_comments  toxic_comments  toxicity_rate
        17            1568             771       0.491709
